# 01 — Gemma 3 270M
## Impact of Code on SLMs
> Model: `google/gemma-3-1b-it` (270M class)

This notebook fine-tunes and evaluates Gemma 3 270M across all dataset mixture experiments.

## Setup & Installs

In [3]:
!pip install transformers datasets peft accelerate lm-eval bitsandbytes tqdm -q

  error: subprocess-exited-with-error
  
  × installing build dependencies for pyarrow did not run successfully.
  │ exit code: 1
  ╰─> [496 lines of output]
        Using cached scikit_build_core-0.12.2-py3-none-any.whl.metadata (16 kB)
        Using cached cython-3.2.5-py3-none-any.whl.metadata (6.9 kB)
        Using cached libcst-1.8.6.tar.gz (891 kB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Preparing metadata (pyproject.toml): started
        Preparing metadata (pyproject.toml): finished with status 'done'
        Using cached numpy-2.4.6-cp315-cp315-macosx_26_0_arm64.whl
        Using cached setuptools_scm-10.0.5-py3-none-any.whl.metadata (6.5 kB)
        Using cached packaging-26.2-py3-none-any.whl.metadata (3.5 kB)
        Using cached pathspec-1.1.1-py3-none-any.whl.metadata

In [ ]:
import os, random, json, subprocess
import torch
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm
from datetime import datetime

# ── Auth ──────────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

# ── Directories ───────────────────────────────────────────────────────────────
DATA_DIR    = "./data"
CACHE_DIR   = "./cache"
RESULTS_DIR = "./results"
for d in [DATA_DIR, CACHE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Global hyper-params ───────────────────────────────────────────────────────
NUM_SAMPLES  = 10_000
MAX_SEQ_LEN  = 512
BATCH_SIZE   = 1
GRAD_ACCUM   = 16        # effective batch = 16
LR           = 2e-4
EPOCHS       = 1
SAVE_STEPS   = 300
LOG_STEPS    = 50
WARMUP_STEPS = 50

# ── Device setup ──────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE  = torch.bfloat16
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE  = torch.float32   # bfloat16 not fully supported on MPS
else:
    DEVICE = "cpu"
    DTYPE  = torch.float32

print(f"Device: {DEVICE}  |  dtype: {DTYPE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif DEVICE == "mps":
    print("GPU: Apple MPS (Metal)")
else:
    print("GPU: CPU only")

# ── Model loading ─────────────────────────────────────────────────────────────
def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=HF_TOKEN,
        cache_dir=CACHE_DIR,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=DTYPE,
        device_map={"": DEVICE},
        token=HF_TOKEN,
        cache_dir=CACHE_DIR,
    )
    return model, tokenizer

# ── LoRA config ───────────────────────────────────────────────────────────────
def apply_lora(model):
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "v_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

# ── Training arguments ────────────────────────────────────────────────────────
def get_training_args(output_dir: str) -> TrainingArguments:
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_steps=WARMUP_STEPS,
        logging_steps=LOG_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        bf16=False,     # not supported on MPS
        fp16=False,     # not supported on MPS
        dataloader_pin_memory=False,  # must be False for MPS
        report_to="none",
    )

# ── Tokenization ──────────────────────────────────────────────────────────────
def tokenize_dataset(dataset, tokenizer):
    def tokenize(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding="max_length",
        )
    return dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)

# ── Training loop ─────────────────────────────────────────────────────────────
def train(model_name: str, dataset, run_name: str):
    print(f"\n{'='*60}")
    print(f" Training: {run_name}")
    print(f" Model:    {model_name}")
    print(f" Samples:  {len(dataset)}")
    print(f"{'='*60}\n")

    model, tokenizer = load_model_and_tokenizer(model_name)
    model = apply_lora(model)

    tokenized = tokenize_dataset(dataset, tokenizer)
    collator  = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    output_dir = os.path.join(RESULTS_DIR, run_name)
    args       = get_training_args(output_dir)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized,
        data_collator=collator,
    )

    start = datetime.now()
    trainer.train()
    elapsed = datetime.now() - start

    print(f"\nFinished {run_name} in {elapsed}")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return trainer

# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    MODEL_NAME = "facebook/opt-125m"   # swap to your SLM of choice

    print("Loading dataset...")
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train", cache_dir=CACHE_DIR)
    raw = raw.select(range(min(NUM_SAMPLES, len(raw))))

    trainer = train(
        model_name=MODEL_NAME,
        dataset=raw,
        run_name=f"slm_run_{datetime.now().strftime('%Y%m%d_%H%M')}",
    )
    print("\nDone. Model saved to:", RESULTS_DIR)

Device: mps  |  dtype: torch.float32
GPU: Apple MPS (Metal)
Loading dataset...


Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 1890503.84 examples/s]



 Training: slm_run_20260609_2008
 Model:    facebook/opt-125m
 Samples:  10000



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 512.67it/s]


trainable params: 589,824 || all params: 125,829,120 || trainable%: 0.4688


Map: 100%|██████████| 10000/10000 [00:00<00:00, 17559.60 examples/s]


Step,Training Loss
50,3.870464
100,3.686325
150,3.549566
200,3.555005
250,3.476216
300,3.471514
350,3.468877
400,3.455095
450,3.470901


## Section 1 — Dataset Preparation

In [10]:
# ═══════════════════════════════════════════════════════════════
#   SECTION 1: DATASET PREPARATION
#   Datasets (from image plan):
#     D1  — NLP only             (10k from SlimPajama)
#     D2  — Code: Python only    (10k from The Stack)
#     D3  — Code: Java only      (10k from The Stack)
#     D4  — Code: C++ only       (10k from The Stack)
#     D5  — Code: Python∪Java∪C++ balanced sample (10k total)
#   Mixture datasets built from D1 + D5:
#     M1  — Text 100%
#     M2  — Code 100% (Python+Java from The Stack)
#     M3  — Text 70% / Code 30%
#     M4  — Code 70% / Text 30%
#     M5  — Text 75% / Code 25%
#     M6  — Code 75% / Text 25%
#     M7  — Code 50% / Text 50%
# ═══════════════════════════════════════════════════════════════

TEXT_SOURCES = [
    "RedPajamaCommonCrawl",
    "RedPajamaC4",
    "RedPajamaBook",
    "RedPajamaArXiv",
    "RedPajamaWikipedia",
]

def stream_samples(iterable, n, desc="Loading"):
    """Stream n samples from a HF streaming dataset."""
    samples = []
    with tqdm(total=n, desc=desc) as pbar:
        for item in iterable:
            txt = item.get("text") or item.get("content") or ""
            if txt.strip():
                samples.append({"text": txt})
                pbar.update(1)
                if len(samples) >= n:
                    break
    return samples


def load_nlp(n=NUM_SAMPLES):
    print(f"\n[D1] Loading NLP text ({n:,} samples)...")
    path = f"{DATA_DIR}/d1_nlp"
    if os.path.exists(path):
        print("  → cache hit"); return Dataset.load_from_disk(path)
    ds = load_dataset("DKYoon/SlimPajama-6B", split="train", streaming=True, cache_dir=CACHE_DIR)
    ds = ds.filter(lambda x: (x.get("meta") or {}).get("redpajama_set_name") in TEXT_SOURCES)
    samples = stream_samples(ds, n, "  NLP text")
    out = Dataset.from_list(samples); out.save_to_disk(path)
    print(f"  ✓ {len(out)} samples saved"); return out


def load_code_lang(lang, n=NUM_SAMPLES):
    """Load code from The Stack for a given language."""
    tag = lang.lower().replace("+", "pp").replace("#", "sharp")
    path = f"{DATA_DIR}/d_code_{tag}"
    if os.path.exists(path):
        print(f"  → cache hit ({lang})"); return Dataset.load_from_disk(path)
    print(f"  Loading {lang} from The Stack...")
    # Map common names to The Stack data_dir names
    stack_lang = {
        "python": "data/python",
        "java":   "data/java",
        "c++":    "data/cpp",
        "cpp":    "data/cpp",
    }.get(lang.lower(), f"data/{lang.lower()}")
    ds = load_dataset(
        "bigcode/the-stack", data_dir=stack_lang,
        split="train", streaming=True, cache_dir=CACHE_DIR, token=HF_TOKEN
    )
    samples = stream_samples(ds, n, f"  {lang}")
    out = Dataset.from_list(samples); out.save_to_disk(path)
    print(f"  ✓ {len(out)} {lang} samples"); return out


def build_all_datasets():
    """Build D1-D5 and all mixture datasets."""
    print("\n" + "="*60)
    print("  DATASET PREPARATION")
    print("="*60)

    # Base datasets
    d1_nlp  = load_nlp(NUM_SAMPLES)
    d2_py   = load_code_lang("python",  NUM_SAMPLES)
    d3_java = load_code_lang("java",    NUM_SAMPLES)
    d4_cpp  = load_code_lang("c++",     NUM_SAMPLES)

    # D5: Python ∪ Java ∪ C++ balanced (10k total → ~3333 each)
    path_d5 = f"{DATA_DIR}/d5_union"
    if os.path.exists(path_d5):
        d5_union = Dataset.load_from_disk(path_d5)
        print(f"  [D5] cache hit: {len(d5_union)} union samples")
    else:
        n3 = NUM_SAMPLES // 3
        sub_py   = d2_py.select(range(min(n3, len(d2_py))))
        sub_java = d3_java.select(range(min(n3, len(d3_java))))
        sub_cpp  = d4_cpp.select(range(min(n3, len(d4_cpp))))
        d5_union = concatenate_datasets([sub_py, sub_java, sub_cpp])
        d5_union = d5_union.shuffle(seed=42).select(range(min(NUM_SAMPLES, len(d5_union))))
        d5_union.save_to_disk(path_d5)
        print(f"  [D5] Built union: {len(d5_union)} samples")

    # Code for mixtures = Python + Java (as specified in user prompt)
    path_pj = f"{DATA_DIR}/d_py_java"
    if os.path.exists(path_pj):
        d_py_java = Dataset.load_from_disk(path_pj)
    else:
        half = NUM_SAMPLES // 2
        sub_py2   = d2_py.select(range(min(half, len(d2_py))))
        sub_java2 = d3_java.select(range(min(half, len(d3_java))))
        d_py_java = concatenate_datasets([sub_py2, sub_java2]).shuffle(seed=42)
        d_py_java.save_to_disk(path_pj)
    print(f"  [Code base for mixtures] Python+Java: {len(d_py_java)} samples")

    # Build mixture datasets
    ratios = {
        "m1_text100":   (1.00, 0.00),
        "m2_code100":   (0.00, 1.00),
        "m3_t70_c30":   (0.70, 0.30),
        "m4_c70_t30":   (0.30, 0.70),
        "m5_t75_c25":   (0.75, 0.25),
        "m6_c75_t25":   (0.25, 0.75),
        "m7_c50_t50":   (0.50, 0.50),
    }

    mixture_datasets = {}
    for name, (t_frac, c_frac) in ratios.items():
        path_m = f"{DATA_DIR}/{name}"
        if os.path.exists(path_m):
            mixture_datasets[name] = Dataset.load_from_disk(path_m)
            print(f"  [{name}] cache hit: {len(mixture_datasets[name])} samples")
            continue

        t_n = int(NUM_SAMPLES * t_frac)
        c_n = NUM_SAMPLES - t_n

        parts = []
        if t_n > 0:
            parts.append(d1_nlp.shuffle(seed=42).select(range(min(t_n, len(d1_nlp)))))
        if c_n > 0:
            parts.append(d_py_java.shuffle(seed=42).select(range(min(c_n, len(d_py_java)))))

        combined = concatenate_datasets(parts).shuffle(seed=42) if len(parts) > 1 else parts[0]
        combined.save_to_disk(path_m)
        mixture_datasets[name] = combined
        print(f"  [{name}] text={t_n} code={c_n} total={len(combined)}")

    return d1_nlp, d2_py, d3_java, d4_cpp, d5_union, mixture_datasets


# Build all datasets
d1_nlp, d2_py, d3_java, d4_cpp, d5_union, mixture_datasets = build_all_datasets()
print("\n✓ All datasets ready.")



  DATASET PREPARATION

[D1] Loading NLP text (10,000 samples)...
  → cache hit
  Loading python from The Stack...


DatasetNotFoundError: Dataset 'bigcode/the-stack' is a gated dataset on the Hub. You must be authenticated to access it.

## Section 2 — Training Utilities

In [1]:
# ═══════════════════════════════════════════════════════════════
#   SECTION 2: TRAINING UTILITIES
# ═══════════════════════════════════════════════════════════════

def load_base_model(model_id, lora=False):
    """Load model + tokenizer; optionally wrap with LoRA."""
    print(f"\nLoading {model_id} (LoRA={lora})...")
    tok = AutoTokenizer.from_pretrained(
        model_id, cache_dir=CACHE_DIR, token=HF_TOKEN, trust_remote_code=True
    )
    tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id, cache_dir=CACHE_DIR, token=HF_TOKEN,
        torch_dtype=DTYPE, device_map="auto" if DEVICE == "cuda" else None,
        trust_remote_code=True
    )

    if lora:
        lora_cfg = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=8, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"],
            bias="none",
        )
        model = get_peft_model(model, lora_cfg)
        model.print_trainable_parameters()

    return model, tok


def tokenize_ds(dataset, tokenizer):
    def _tok(examples):
        return tokenizer(
            examples["text"], truncation=True,
            max_length=MAX_SEQ_LEN, padding=False
        )
    return dataset.map(_tok, batched=True, remove_columns=dataset.column_names, desc="Tokenizing")


def fine_tune(model_id, dataset, output_dir, lora=False):
    """Fine-tune a model on a given dataset."""
    if os.path.exists(output_dir) and os.path.exists(f"{output_dir}/config.json"):
        print(f"  ↳ Already trained: {output_dir}")
        return

    model, tok = load_base_model(model_id, lora=lora)
    tokenized  = tokenize_ds(dataset, tok)
    split      = tokenized.train_test_split(test_size=0.1, seed=42)

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        bf16=torch.cuda.is_available(),
        logging_steps=LOG_STEPS,
        eval_strategy="steps",
        eval_steps=SAVE_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=1,
        report_to="none",
        gradient_checkpointing=True,
        warmup_steps=WARMUP_STEPS,
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=split["train"],
        eval_dataset=split["test"],
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
    )

    print(f"\n  ── Training → {output_dir}")
    trainer.train()
    trainer.save_model(output_dir)
    tok.save_pretrained(output_dir)
    del model; torch.cuda.empty_cache()
    print(f"  ✓ Saved → {output_dir}")


## Section 3 — Evaluation Utilities

In [2]:
# ═══════════════════════════════════════════════════════════════
#   SECTION 3: EVALUATION (lm-evaluation-harness)
#   Benchmarks from the paper:
#     NL Reasoning : hellaswag, boolq, arc_easy
#     World Knowledge: triviaqa, nq_open
#     Code          : humaneval, mbpp
# ═══════════════════════════════════════════════════════════════

NL_TASKS    = ["hellaswag", "boolq", "arc_easy"]
WK_TASKS    = ["triviaqa"]           # nq_open requires extra deps; triviaqa standard
CODE_TASKS  = ["humaneval"]          # mbpp can be added if harness version supports it

ALL_TASKS   = NL_TASKS + WK_TASKS + CODE_TASKS

# Install lm-evaluation-harness if needed
def install_lm_eval():
    try:
        import lm_eval
        print("lm-eval already installed")
    except ImportError:
        print("Installing lm-evaluation-harness...")
        subprocess.run(
            ["pip", "install", "lm-eval[all]", "--break-system-packages", "-q"],
            check=True
        )
        print("✓ lm-eval installed")

install_lm_eval()


def run_lm_eval(model_path_or_id, label, tasks=ALL_TASKS):
    """Run lm-evaluation-harness and return dict of scores."""
    print(f"\n── Evaluating: {label}")
    results = {}

    for task in tasks:
        out_path = f"{RESULTS_DIR}/{label}_{task}.json"
        if os.path.exists(out_path):
            with open(out_path) as f:
                data = json.load(f)
            results[task] = _extract_score(data, task)
            print(f"   ↳ {task}: {results[task]:.4f} (cached)")
            continue

        cmd = [
            "python", "-m", "lm_eval",
            "--model", "hf",
            "--model_args", f"pretrained={model_path_or_id},trust_remote_code=True",
            "--tasks", task,
            "--device", DEVICE,
            "--batch_size", "4",
            "--num_fewshot", "0",
            "--output_path", out_path,
        ]
        try:
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
            if proc.returncode != 0:
                print(f"   ⚠ {task} failed:\n{proc.stderr[-500:]}")
                results[task] = None
                continue
            if os.path.exists(out_path):
                with open(out_path) as f:
                    data = json.load(f)
                results[task] = _extract_score(data, task)
                print(f"   ✓ {task}: {results[task]:.4f}")
            else:
                results[task] = None
        except subprocess.TimeoutExpired:
            print(f"   ⚠ {task} timed out")
            results[task] = None
        except Exception as e:
            print(f"   ⚠ {task} error: {e}")
            results[task] = None

    return results


def _extract_score(data, task):
    METRIC = {
        "hellaswag": "acc_norm,none",
        "boolq":     "acc,none",
        "arc_easy":  "acc_norm,none",
        "triviaqa":  "exact_match,remove_whitespace",
        "humaneval": "pass@1",
        "mbpp":      "pass@1",
    }
    try:
        r = data.get("results", {}).get(task, {})
        m = METRIC.get(task, "acc,none")
        return float(r.get(m, r.get("acc,none", 0.0)))
    except Exception:
        return 0.0


lm-eval already installed


## Section 4 — Fine-Tuning: Gemma 3 270M × All Datasets

In [3]:
# ═══════════════════════════════════════════════════════════════
#   SECTION 4: TRAIN ALL EXPERIMENTS — google/gemma-3-1b-it
#   25 models total (image plan):
#     NFT   (0): baseline, no fine-tuning (just eval)
#     NFT-LoRA: baseline with LoRA unfrozen (still NF but with adapter init?)
#     Experiments per dataset mixture (7 datasets × model = 7 FT + 1 LoRA)
# ═══════════════════════════════════════════════════════════════

MODEL_ID  = "google/gemma-3-1b-it"
MODEL_TAG = "gemma3_270m"
BASE_OUT  = f"./models/gemma3_270m"

# Dataset name → Dataset object (built above)
DATASET_MAP = {
    "m1_text100":   mixture_datasets["m1_text100"],
    "m2_code100":   mixture_datasets["m2_code100"],
    "m3_t70_c30":   mixture_datasets["m3_t70_c30"],
    "m4_c70_t30":   mixture_datasets["m4_c70_t30"],
    "m5_t75_c25":   mixture_datasets["m5_t75_c25"],
    "m6_c75_t25":   mixture_datasets["m6_c75_t25"],
    "m7_c50_t50":   mixture_datasets["m7_c50_t50"],
    # Plus the 3 pure-language code datasets (D2-D4) and union (D5)
    "d2_python":    d2_py,
    "d3_java":      d3_java,
    "d4_cpp":       d4_cpp,
    "d5_union":     d5_union,
}

# ─── NFT baseline (no training — evaluate base model directly) ────────────────
print("\n" + "="*60)
print(f"  GEMMA3_270M — EXPERIMENT PLAN")
print("="*60)
print("  NFT  : base model (no fine-tuning)")
print("  + 7 mixture experiments (full fine-tune)")
print("  + 1 LoRA experiment (m1_text100 + LoRA)")
print("  + 4 pure-code experiments (python, java, c++, union)")
print("  Total fine-tuned checkpoints: 12")

nft_label  = f"{MODEL_TAG}_nft"
nft_path   = MODEL_ID         # Use HF hub ID for evaluation (no training)

# ─── Full fine-tune experiments ───────────────────────────────────────────────
ft_experiments = list(DATASET_MAP.keys())

for ds_name in ft_experiments:
    out_dir = f"{BASE_OUT}/{ds_name}"
    print(f"\n[FT] {MODEL_TAG} × {ds_name}")
    fine_tune(MODEL_ID, DATASET_MAP[ds_name], out_dir, lora=False)

# ─── LoRA experiment (on m1_text100 as per paper NFT+LoRA setup) ──────────────
lora_out = f"{BASE_OUT}/lora_m1_text100"
print(f"\n[LoRA] {MODEL_TAG} × m1_text100 (LoRA)")
fine_tune(MODEL_ID, mixture_datasets["m1_text100"], lora_out, lora=True)

print(f"\n✓ All {MODEL_TAG} fine-tuning complete.")


NameError: name 'mixture_datasets' is not defined

## Section 5 — Evaluate All Gemma 3 Checkpoints

In [4]:
# ═══════════════════════════════════════════════════════════════
#   SECTION 5: EVALUATE ALL GEMMA3_270M CHECKPOINTS
# ═══════════════════════════════════════════════════════════════

MODEL_TAG = "gemma3_270m"
MODEL_ID  = "google/gemma-3-1b-it"
BASE_OUT  = f"./models/{MODEL_TAG}"

checkpoint_map = {
    f"{MODEL_TAG}_nft":          MODEL_ID,
    f"{MODEL_TAG}_m1_text100":   f"{BASE_OUT}/m1_text100",
    f"{MODEL_TAG}_m2_code100":   f"{BASE_OUT}/m2_code100",
    f"{MODEL_TAG}_m3_t70_c30":   f"{BASE_OUT}/m3_t70_c30",
    f"{MODEL_TAG}_m4_c70_t30":   f"{BASE_OUT}/m4_c70_t30",
    f"{MODEL_TAG}_m5_t75_c25":   f"{BASE_OUT}/m5_t75_c25",
    f"{MODEL_TAG}_m6_c75_t25":   f"{BASE_OUT}/m6_c75_t25",
    f"{MODEL_TAG}_m7_c50_t50":   f"{BASE_OUT}/m7_c50_t50",
    f"{MODEL_TAG}_d2_python":    f"{BASE_OUT}/d2_python",
    f"{MODEL_TAG}_d3_java":      f"{BASE_OUT}/d3_java",
    f"{MODEL_TAG}_d4_cpp":       f"{BASE_OUT}/d4_cpp",
    f"{MODEL_TAG}_d5_union":     f"{BASE_OUT}/d5_union",
    f"{MODEL_TAG}_lora":         f"{BASE_OUT}/lora_m1_text100",
}

all_results_{model_tag} = {}
for label, path in checkpoint_map.items():
    all_results_{model_tag}[label] = run_lm_eval(path, label)

# Save per-model results
with open(f"{RESULTS_DIR}/{MODEL_TAG}_results.json", "w") as f:
    json.dump(all_results_{model_tag}, f, indent=2)
print(f"\n✓ Results saved → {RESULTS_DIR}/{MODEL_TAG}_results.json")


SyntaxError: invalid syntax (1980248990.py, line 25)